# Concrete Compressive Strength（混凝土抗压强度）

In [ ]:
from pathlib import Path

# 工具入参要求绝对路径；本 cell 算好后经 {VAR} 注入下方各命令
HERE = Path.cwd()
PY = str(HERE / "concrete.py")      # :load / :load_X / :load_X_to_predict 按需拼后缀
YAML = str(HERE / "concrete.yaml")


In [ ]:
# ① 数据画像（concrete.py:load 全量数据）
!uv --project ../.. run --extra data mflowy data_profile '{"file_path": "{PY}:load"}'

In [ ]:
# ② 探索性数据分析（concrete.py:load 全量数据）
!uv --project ../.. run --extra data mflowy eda '{"file_path": "{PY}:load","target":"strength"}'

In [ ]:
# ③ 多模型对比（从输出中记录 XGB 步骤的 run_id）
!uv --project ../.. run --extra model --extra deep mflowy modeling '{"modeling_steps_yaml": "{YAML}", "name": "concrete-v1", "desc": "Concrete strength modeling"}'

In [ ]:
# ④ SHAP 解释（把下方 63b17b33d7b943e7b74c0b8b798d1a1b 替换为 ③ 输出的裸 run_id（无 XGB= 前缀））
!uv --project ../.. run --extra model --extra deep mflowy explanation '{"modeling_steps_yaml": "{YAML}", "model": "63b17b33d7b943e7b74c0b8b798d1a1b", "name": "concrete-shap", "desc": "Concrete SHAP"}'

In [ ]:
# ⑤ 逆向搜索：目标强度最大化（concrete.py:load_X 提供搜索空间，替代 X.csv）
!uv --project ../.. run --extra model --extra deep mflowy inverse_optimization '{"data": "{PY}:load_X", "model": "63b17b33d7b943e7b74c0b8b798d1a1b", "direction": {"strength": "maximize"}, "constraint": {"cement": {"start": 102, "end": 540}, "water": {"start": 122, "end": 247}}, "n_trials": 200}'

In [ ]:
# ⑥ 上线前预测验证（concrete.py:load_X_to_predict 提供推荐配比单行，替代 recommended.csv）
!uv --project ../.. run --extra model --extra deep mflowy predict '{"data": "{PY}:load_X_to_predict", "model": "63b17b33d7b943e7b74c0b8b798d1a1b"}'